# 15 — RQ3: đánh giá Candidate Requirement (máy local)

**Phạm vi**

Đánh giá 16 Candidate Requirement của GĐ14 theo ba khía cạnh của RQ3: structural quality (tự động), semantic correctness (chấm tay) và
source traceability (tự động và chấm tay). Cách đánh giá chốt trước khi chấm (amendment v2.9). Output GĐ14 không được sửa.

**Hai phần**

Phần 1 (tự động) chạy ngay; phần 2 tạo phiếu chấm và chỉ tính điểm khi phiếu đã được điền (`07_human_eval/rq3_requirements/`).
Không gọi API trả phí.

In [1]:
import os, sys, json, datetime, subprocess
import numpy as np, pandas as pd, yaml
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.utils.display import show as show_table
from src.generation import core as G
from src.evaluation import core as E
RQ2 = os.path.join(ROOT, '05_experiments', 'rq2_topics')
RQ3 = os.path.join(ROOT, '05_experiments', 'rq3_requirements')
HE = os.path.join(ROOT, '07_human_eval', 'rq3_requirements')
os.makedirs(HE, exist_ok=True)
COMMIT = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Ngày chạy:', datetime.date.today().strftime('%d/%m/%Y'), '| commit:', COMMIT)

Ngày chạy: 02/11/2026 | commit: 78b1838b7c59c4a0dec8ea8a70dce31529f2552f


## Dữ liệu

**Đầu vào**

`requirements.csv`, `requirement_evidence.csv`, `gd14_generation_units.csv`, `gd14_evidence_pool.csv` và log lượt gọi
`gd14_generation_calls.jsonl` (review_id đã đưa vào prompt của từng unit, lấy từ lần gọi cuối). Văn bản review đọc từ
`actionable_reviews.csv` (không commit), chỉ dùng cho phiếu chấm.

In [2]:
REQ = pd.read_csv(os.path.join(RQ3, 'requirements.csv'), encoding='utf-8-sig')
EV = pd.read_csv(os.path.join(RQ3, 'requirement_evidence.csv'))
UNITS = pd.read_csv(os.path.join(RQ3, 'gd14_generation_units.csv'), encoding='utf-8-sig', dtype={'unit_id': str})
POOL = pd.read_csv(os.path.join(RQ3, 'gd14_evidence_pool.csv'), dtype={'unit_id': str})
MAP = pd.read_csv(os.path.join(RQ2, 'review_topic_map.csv'))
ACT = pd.read_csv(os.path.join(RQ2, 'actionable_reviews.csv'), encoding='utf-8-sig')
TEXT = ACT.set_index('review_id')['text']
CALLS = [json.loads(l) for l in open(os.path.join(RQ3, 'gd14_generation_calls.jsonl'), encoding='utf-8')]
SHOWN = {c['unit_id']: (None, c['shown_review_ids']) for c in CALLS}
GEN = pd.read_json(os.path.join(RQ3, 'gd14_generation.json'), orient='records', dtype={'unit_id': str})
FZ = yaml.safe_load(open(os.path.join(ROOT, '03_configs', 'frozen', 'rq3_generation_config.yaml'), encoding='utf-8'))['rq3_generation']
assert set(GEN['prompt_hash']) == {FZ['prompt_hash']}, 'output không sinh từ prompt đã khóa'
print('Requirement:', len(REQ), '| liên kết evidence:', len(EV), '| unit đủ điều kiện:', int(UNITS['eligible'].sum()),
      '| theo loại:', REQ['type'].value_counts().to_dict())

Requirement: 16 | liên kết evidence: 93 | unit đủ điều kiện: 16 | theo loại: {'BUG': 7, 'UX': 6, 'FEATURE': 3}


## Phần 1 — đánh giá tự động

**Luật QUS (theo AQUSA)**

Sáu loại lỗi mà AQUSA phát hiện tự động (Lucassen et al., 2016), cài đặt lại trong `src/evaluation/core.py`: *well-formed* (có role và
means), *atomic* (means hoặc ends nối hai ý bằng "and", "or", "&", "/"), *minimal* (ngoặc, "e.g.", "etc.", dấu "…" hoặc nhiều câu),
*explicit dependencies* (tham chiếu story khác), *uniform* (khác khung mẫu phổ biến nhất theo vị trí: role | means | ends), *unique*
(trùng nguyên văn hoặc trùng means). Đây là bản cài đặt lại các luật, không phải công cụ AQUSA gốc.

In [3]:
QUS = E.qus_defects(REQ)
QUS_SUM = E.qus_summary(QUS)
show_table(QUS_SUM, 'Lỗi QUS của User Story (luật theo AQUSA)', percent=['defect_rate'])
show_table(QUS[QUS['n_defects'] > 0][['requirement_id', 'role', 'means', 'ends', *E.QUS_RULES]], 'User Story có lỗi QUS',
           wrap=['means', 'ends'])
show_table(QUS['role'].value_counts().rename_axis('role').reset_index(name='số story'), 'Role trong User Story')

rule,n_defect,n_story,defect_rate
well_formed,0,16,"0,0%"
atomic,1,16,"6,2%"
minimal,0,16,"0,0%"
explicit_dependencies,0,16,"0,0%"
uniform,0,16,"0,0%"
unique,0,16,"0,0%"
any,1,16,"6,2%"


requirement_id,role,means,ends,well_formed,atomic,minimal,explicit_dependencies,uniform,unique
REQ-07-BUG-1,shopper,to be able to load and place orders in the app,I can purchase items without issues,False,True,False,False,False,False


role,số story
shopper,9
buyer,5
gamer,2


**Acceptance Criteria, sinh hợp lệ và truy vết**

AC phải ở dạng "Given …, when …, then …." và mỗi requirement có 2–3 AC (amendment v2.7). Valid generation rate: tỷ lệ unit cho output
qua mọi kiểm tra ở lần gọi đầu và sau một lần gọi lại. Evidence-ID validity dùng lại kiểm tra backend của GĐ14 trên danh sách review đã đưa
vào prompt (lấy từ log). Traceability coverage: tỷ lệ requirement có ít nhất một evidence hợp lệ, và tỷ lệ unit đủ điều kiện có requirement.

In [4]:
AC_CRIT, AC_REQ = E.ac_validity(REQ, tuple(FZ['output']['acceptance_criteria']))
first = pd.DataFrame([c for c in CALLS if c['attempt'] == 1])
CHECK, EV_OK = G.verify_evidence(REQ, EV, POOL, ACT, MAP, UNITS, SHOWN)
elig = set(UNITS.loc[UNITS['eligible'], 'unit_id'])
AUTO = pd.DataFrame([
    {'chỉ số': 'AC đúng dạng Given/When/Then', 'đạt': int(AC_CRIT['gwt_valid'].sum()), 'tổng': len(AC_CRIT)},
    {'chỉ số': 'requirement có 2–3 AC', 'đạt': int(AC_REQ['count_in_range'].sum()), 'tổng': len(AC_REQ)},
    {'chỉ số': 'valid generation — lần gọi đầu', 'đạt': int(first['issues'].map(lambda x: not x).sum()), 'tổng': len(first)},
    {'chỉ số': 'valid generation — sau một lần gọi lại', 'đạt': int(GEN['issues'].map(lambda x: not x).sum()), 'tổng': len(GEN)},
    {'chỉ số': 'liên kết evidence hợp lệ (cả 4 kiểm tra)', 'đạt': int(CHECK['đạt'].min()), 'tổng': int(CHECK['tổng'].max())},
    {'chỉ số': 'requirement có ≥ 1 evidence hợp lệ', 'đạt': int(REQ['requirement_id'].isin(EV['requirement_id']).sum()), 'tổng': len(REQ)},
    {'chỉ số': 'unit đủ điều kiện có requirement', 'đạt': len(elig & set(REQ['unit_id'].astype(str))), 'tổng': len(elig)},
    {'chỉ số': 'User Story không có lỗi QUS', 'đạt': int((QUS['n_defects'] == 0).sum()), 'tổng': len(QUS)},
])
AUTO['tỷ lệ'] = AUTO['đạt'] / AUTO['tổng']
show_table(AUTO, 'Đánh giá tự động: cấu trúc và truy vết', percent=['tỷ lệ'])
show_table(CHECK, 'Kiểm tra evidence ID (đọc lại từ log)')
show_table(REQ.groupby('type')['n_evidence_cited'].describe()[['count', 'mean', 'min', 'max']].reset_index(),
           'Số evidence được trích mỗi requirement', decimals=1)

chỉ số,đạt,tổng,tỷ lệ
AC đúng dạng Given/When/Then,47,47,"100,0%"
requirement có 2–3 AC,16,16,"100,0%"
valid generation — lần gọi đầu,15,16,"93,8%"
valid generation — sau một lần gọi lại,16,16,"100,0%"
liên kết evidence hợp lệ (cả 4 kiểm tra),93,93,"100,0%"
requirement có ≥ 1 evidence hợp lệ,16,16,"100,0%"
unit đủ điều kiện có requirement,16,16,"100,0%"
User Story không có lỗi QUS,15,16,"93,8%"


kiểm tra,đạt,tổng,hợp lệ
tồn tại trong corpus actionable,93,93,True
thuộc đúng cụm,93,93,True
thuộc tập evidence của unit,93,93,True
nằm trong danh sách đưa vào prompt,93,93,True


type,count,mean,min,max
BUG,"7,0","5,3","4,0","7,0"
FEATURE,"3,0","4,3","4,0","5,0"
UX,"6,0","7,2","5,0","10,0"


**User Story gần trùng**

Cosine giữa embedding của các User Story (mô hình mpnet đã pin ở GĐ12), báo cặp có cosine ≥ 0,85. Chỉ báo cáo cho BA, không loại
requirement (amendment v2.9); luật *unique* ở trên chỉ bắt trùng nguyên văn.

In [5]:
from sentence_transformers import SentenceTransformer
FZ12 = yaml.safe_load(open(os.path.join(ROOT, '03_configs', 'frozen', 'rq2_topics_config.yaml'), encoding='utf-8'))['rq2_topics']
st = SentenceTransformer(FZ12['embedding']['model_name'], revision=FZ12['embedding']['revision'])
US_EMB = st.encode(REQ['user_story'].tolist(), normalize_embeddings=True, show_progress_bar=False)
DUP = E.near_duplicates(REQ, US_EMB, 0.85)
show_table(DUP.merge(REQ[['requirement_id', 'user_story']].rename(columns={'requirement_id': 'requirement_a', 'user_story': 'story_a'}))
              .merge(REQ[['requirement_id', 'user_story']].rename(columns={'requirement_id': 'requirement_b', 'user_story': 'story_b'})),
           'Cặp User Story gần trùng (cosine ≥ 0,85)', wrap=['story_a', 'story_b'])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

requirement_a,requirement_b,cosine,cluster_a,cluster_b,story_a,story_b
REQ-03-1,REQ-05-1,"0,988",3,5,"As a shopper, I want fewer ads, so that I can use the app without interruptions.","As a shopper, I want fewer intrusive ads, so that I can use the app without interruptions."
REQ-10-1,REQ-14-1,"0,866",10,14,"As a gamer, I want to avoid being interrupted by ads, so that I can enjoy my gaming experience.","As a gamer, I want to avoid interruptions from notifications while playing games, so that I can enjoy my gaming experience without disruptions."
REQ-05-1,REQ-15-1,"0,860",5,15,"As a shopper, I want fewer intrusive ads, so that I can use the app without interruptions.","As a shopper, I want fewer ads from Shopee on other platforms, so that I can enjoy my browsing experience without interruptions."
REQ-03-1,REQ-15-1,"0,857",3,15,"As a shopper, I want fewer ads, so that I can use the app without interruptions.","As a shopper, I want fewer ads from Shopee on other platforms, so that I can enjoy my browsing experience without interruptions."


In [6]:
QUS.to_csv(os.path.join(RQ3, 'gd15_qus_defects.csv'), index=False)
QUS_SUM.to_csv(os.path.join(RQ3, 'gd15_qus_summary.csv'), index=False)
AC_CRIT.to_csv(os.path.join(RQ3, 'gd15_ac_validity.csv'), index=False)
AUTO.to_csv(os.path.join(RQ3, 'gd15_auto_summary.csv'), index=False, encoding='utf-8-sig')
DUP.to_csv(os.path.join(RQ3, 'gd15_near_duplicates.csv'), index=False)
print('Đã lưu: gd15_qus_defects.csv, gd15_qus_summary.csv, gd15_ac_validity.csv, gd15_auto_summary.csv, gd15_near_duplicates.csv')

Đã lưu: gd15_qus_defects.csv, gd15_qus_summary.csv, gd15_ac_validity.csv, gd15_auto_summary.csv, gd15_near_duplicates.csv


**Nhận xét phần tự động**

Về cấu trúc, output đạt gần như trọn vẹn: 15/16 User Story không có lỗi QUS nào, lỗi duy nhất là *atomic* ở REQ-07-BUG-1 ("load and place
orders" gộp hai hành động); cả 47 AC đúng dạng Given/When/Then và mọi requirement có 2–3 AC. 15/16 unit cho output hợp lệ ngay lần gọi đầu,
unit còn lại (cụm 10, trích cả 20 review, vượt giới hạn 10) hợp lệ sau một lần gọi lại. Về truy vết, cả 93 liên kết evidence qua bốn kiểm tra
(tồn tại, đúng cụm, đúng tập evidence của unit, nằm trong danh sách đưa vào prompt) và cả 16 unit đủ điều kiện đều có requirement; phần này
được bảo đảm bởi thiết kế (nhãn evidence ràng buộc bằng enum của JSON schema, ánh xạ ngược ở backend) chứ không phụ thuộc vào việc model
tuân thủ. Về trùng lặp, 4 cặp User Story gần trùng đều thuộc nhóm quảng cáo (cụm 3, 5, 15 và cặp cụm 10–14 về quảng cáo hay thông báo khi
chơi game), đúng với ghi chú trùng chủ đề đã đặt ở GĐ14; requirement không bị gộp tự động mà để BA quyết định.

Giới hạn: đạt luật cấu trúc không có nghĩa requirement đúng nội dung. Các kiểm tra trên không cho biết review được trích có thật sự nêu
vấn đề của requirement không, hay AC có bịa chi tiết không; hai điểm này được đo bằng chấm tay ở phần 2.

## Phần 2 — chấm tay (semantic correctness và độ bám evidence)

**Phiếu chấm**

Toàn bộ 16 requirement (amendment v2.9), hai phiếu trong `07_human_eval/rq3_requirements/` (hướng dẫn và mốc chấm ở `README.md`):
`rq3_requirement_eval_blank.csv` (fidelity, actionability, AC testability 1–5; hallucination, type_correct 0/1; kèm văn bản các review được
trích) và `rq3_evidence_eval_blank.csv` (93 dòng, `supports` 0/1). Phiếu xếp theo requirement_id; không ghi đè nếu đã có.

In [7]:
SHEET_ORDER = REQ.sort_values('requirement_id', ignore_index=True)
paths, created = E.write_sheets(SHEET_ORDER, EV, TEXT, HE)
print('Đã tạo phiếu chấm:' if created else 'Phiếu chấm đã có, không ghi đè:', [os.path.relpath(p, ROOT) for p in paths.values()])

Đã tạo phiếu chấm: ['07_human_eval\\rq3_requirements\\rq3_requirement_eval_blank.csv', '07_human_eval\\rq3_requirements\\rq3_evidence_eval_blank.csv']


**Kết quả chấm tay**

Tính khi có ít nhất một bộ phiếu đã chấm đủ (`_A`, `_B`). Điểm mỗi requirement là trung bình các người chấm; evidence precision = tỷ lệ
review được trích nêu đúng vấn đề. Hai người chấm → weighted κ (quadratic) cho thang 1–5 và Cohen κ cho 0/1; một người → expert
validation exploratory.

In [8]:
SHEETS = E.read_sheets(HE)
if not SHEETS:
    print('Chưa có phiếu đã chấm đủ trong', os.path.relpath(HE, ROOT), '— chấm xong rồi chạy lại từ đây.')
else:
    print('Người chấm:', list(SHEETS), '(expert validation exploratory)' if len(SHEETS) == 1 else '')
    HSUM, HPER, HAGREE = E.score(SHEETS, SHEET_ORDER)
    show_table(HSUM, 'Kết quả chấm tay 16 Candidate Requirement', decimals=2, percent=['share_≥4'])
    if HAGREE is not None:
        show_table(HAGREE, 'Độ đồng thuận giữa hai người chấm', decimals=2, percent=['exact_agreement'])
    show_table(HPER.groupby('type')[[*E.REQ_SCALES, *E.REQ_BINARY, 'evidence_precision']].mean().reset_index(),
               'Theo loại requirement', decimals=2)
    show_table(HPER.groupby('mixed_origin_flag')[[*E.REQ_SCALES, 'hallucination', 'evidence_precision']].mean().reset_index(),
               'Theo cờ mixed-origin (cụm đã lọc so với cụm bình thường)', decimals=2)
    show_table(HPER[['requirement_id', 'type', *E.REQ_SCALES, *E.REQ_BINARY, 'evidence_precision']], 'Điểm từng requirement', decimals=2)
    HSUM.to_csv(os.path.join(RQ3, 'gd15_human_summary.csv'), index=False, encoding='utf-8-sig')
    HPER.to_csv(os.path.join(RQ3, 'gd15_human_per_requirement.csv'), index=False)
    if HAGREE is not None:
        HAGREE.to_csv(os.path.join(RQ3, 'gd15_human_agreement.csv'), index=False)

Chưa có phiếu đã chấm đủ trong 07_human_eval\rq3_requirements — chấm xong rồi chạy lại từ đây.


## Thông tin lượt chạy

`gd15_run_info.json`: ngày chạy, commit, số requirement, kết quả tự động, người chấm đã có.

In [9]:
INFO = {'run_date': datetime.date.today().isoformat(), 'commit': COMMIT, 'n_requirements': len(REQ), 'n_evidence_links': len(EV),
        'generation_prompt_hash': FZ['prompt_hash'], 'auto': AUTO.to_dict(orient='records'),
        'qus': QUS_SUM.to_dict(orient='records'), 'near_duplicate_pairs': len(DUP),
        'raters': list(SHEETS), 'human_eval_done': bool(SHEETS)}
json.dump(INFO, open(os.path.join(RQ3, 'gd15_run_info.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=str)
print('Đã lưu gd15_run_info.json')

Đã lưu gd15_run_info.json
